In [ ]:
# import numpy as np
# import sympy as sp
# from sklearn.base import BaseEstimator, TransformerMixin

# # ── Symbolic derivative cache ──────────────────────────────────────────────────

# _deriv_cache = {}

# def _get_derivatives(N: int, kappa: float):
#     """
#     Symbolically compute and cache the first N derivatives of
#         gamma_tilde(s) = (kappa^2 + s^2)^{-1}
#     returning a list of N numpy-callable functions.
#     """
#     key = (N, kappa)
#     if key in _deriv_cache:
#         return _deriv_cache[key]

#     s = sp.Symbol('s', real=True)
#     expr = 1 / (sp.Rational(kappa).limit_denominator(1000)**2 + s**2)
#     fns = []
#     for _ in range(N):
#         fns.append(sp.lambdify(s, sp.simplify(expr), modules='numpy'))
#         expr = sp.diff(expr, s)

#     _deriv_cache[key] = fns
#     return fns


# # ── Core helpers ───────────────────────────────────────────────────────────────

# def _mat_sqrt_inv(X: np.ndarray) -> np.ndarray:
#     """Compute X^{-1/2} via eigendecomposition."""
#     L, V = np.linalg.eigh(X)
#     return V * (1.0 / np.sqrt(np.clip(L, 1e-12, None)))[None, :] @ V.T

# def _cauchy_kernel_single(X: np.ndarray, Y: np.ndarray, kappa: float) -> float:
#     x = _mat_sqrt_inv(X) @ Y @ _mat_sqrt_inv(X)
#     N = x.shape[0]

#     rho = np.sort(np.clip(np.linalg.eigvalsh(x), 1e-12, None))
#     s = np.log(rho)                                       # s_i = log(rho_i)

#     # ── Vandermonde in s-space: V(s) = prod_{l > k} (s_l - s_k) ──────────────
#     log_abs_V = 0.0
#     for k in range(N):
#         for l in range(k + 1, N):
#             diff = s[l] - s[k]
#             if abs(diff) < 1e-10:
#                 diff = 1e-10
#             log_abs_V += np.log(abs(diff))

#     # ── det(x)^{(N-1)/2} = exp((N-1)/2 * sum(s)) ─────────────────────────────
#     log_det_power = ((N - 1) / 2.0) * np.sum(s)

#     # ── Derivative matrix: M[k, l] = -gamma_tilde^{(k)}(s_l) ─────────────────
#     derivs = _get_derivatives(N, kappa)
#     M = np.array([[-float(derivs[k](s[l])) for l in range(N)]
#                   for k in range(N)])

#     # Row-normalise before slogdet to handle varying derivative magnitudes
#     row_max = np.max(np.abs(M), axis=1)
#     row_max = np.where(row_max == 0, 1.0, row_max)
#     log_row_scales = np.log(row_max)
#     M_normalised = M / row_max[:, None]

#     sign_M, log_abs_det_M_norm = np.linalg.slogdet(M_normalised)

#     if sign_M == 0:
#         return 0.0

#     log_abs_det_M = log_abs_det_M_norm + np.sum(log_row_scales)

#     # ── Final assembly, entirely in log-space ──────────────────────────────────
#     log_val = log_det_power + log_abs_det_M - log_abs_V

#     if not np.isfinite(log_val):
#         return 0.0

#     return float(np.exp(np.clip(log_val, -700, 700)))
# # ── Scikit-learn transformer ───────────────────────────────────────────────────

# class CauchyGramMatrix(BaseEstimator, TransformerMixin):
#     """
#     Scikit-learn transformer that builds the Gram matrix for the strictly
#     positive-definite Cauchy kernel on the SPD manifold (equation 25).

#         K(X, Y) = f(X^{-1/2} Y X^{-1/2})

#     where f is derived via the Helgason-Fourier (spherical) transform with
#     spectral density gamma(t) = (kappa/2) * exp(-kappa|t|).

#     This kernel is guaranteed PD by Godement's theorem, unlike the naive
#     geodesic substitution k(X,Y) = (kappa^2 + delta^2)^{-l} which fails
#     conditional negative definiteness for matrix dimension N >= 2.

#     Parameters
#     ----------
#     kappa : float, default=1.0
#         Scale parameter. Must be > 0. Larger kappa = broader kernel.

#     Usage (mirrors SteinGramMatrix)
#     --------------------------------
#     pipe = make_pipeline(
#         Covariances(),
#         MicrovoltScaler(),
#         CauchyGramMatrix(kappa=1.0),
#         SVC(kernel='precomputed'),
#     )
#     """

#     def __init__(self, kappa: float = 1.0):
#         if kappa <= 0:
#             raise ValueError(f"kappa must be > 0, got {kappa}")
#         self.kappa = kappa
#         self.X_train_ = None

#     def fit(self, X: np.ndarray, y=None):
#         """
#         Store training covariance matrices.

#         Parameters
#         ----------
#         X : ndarray of shape (n_trials, n_channels, n_channels)
#             Batch of SPD covariance matrices.
#         """
#         self.X_train_ = X
#         return self

#     def transform(self, X: np.ndarray) -> np.ndarray:
#         """
#         Compute the (N, M) kernel matrix between X and the stored training set.

#         Parameters
#         ----------
#         X : ndarray of shape (n_trials, n_channels, n_channels)

#         Returns
#         -------
#         K : ndarray of shape (n_trials, n_train_trials)
#         """
#         N = len(X)
#         M = len(self.X_train_)
#         K = np.zeros((N, M))

#         for i in range(N):
#             for j in range(M):
#                 K[i, j] = _cauchy_kernel_single(X[i], self.X_train_[j], self.kappa)

#         return K

In [3]:
import math
import numpy as np
import sympy as sp
import mpmath as mp

from numpy.linalg import eigvalsh
from scipy.linalg import cholesky, solve_triangular

def chebyshev_U(n, x):
    if n == 0:
        return x * 0 + 1
    if n == 1:
        return 2 * x
    U_nm1 = x * 0 + 1
    U_n = 2 * x
    for _ in range(1, n):
        U_np1 = 2 * x * U_n - U_nm1
        U_nm1, U_n = U_n, U_np1
    return U_n

def cauchy_derivative_closed_form(s, k, kappa):
    s = np.asarray(s, dtype=np.float64)
    r2 = s * s + kappa * kappa
    x = s / np.sqrt(r2)
    return ((-1) ** (k + 1)
            * math.factorial(k - 1)
            * r2 ** (-(k + 1) / 2.0)
            * chebyshev_U(k - 1, x))




In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from scipy.linalg import eigh as scipy_eigh


# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────
# def _log_eigenvalues(X, Y):
#     rho = scipy_eigh(Y, X, eigvals_only=True)
#     return np.log(np.clip(rho, 1e-12, None))

def _cauchy_kernel_single(X, Y, kappa=1.0):
    rho = scipy_eigh(Y, X, eigvals_only=True)
    s   = np.log(np.clip(rho, 1e-12, None))
    return float(np.exp(np.sum(-np.log(1.0 + s**2 / kappa**2)) / len(s)))

# def _cauchy_kernel_single(X, Y, kappa=1.0, drop_sinh=False):
#     s  = _log_eigenvalues(X, Y)
#     N  = len(s)
#     log_term1 = np.sum(2.0 * np.log(kappa) - np.log(kappa**2 + s**2))
#     i_idx, j_idx = np.triu_indices(N, k=1)
#     d = s[j_idx] - s[i_idx]
#     d = d[np.abs(d) > 1e-10]
#     half_d   = d / 2.0
#     log_sinh = np.where(half_d > 20.0, half_d - np.log(2.0), np.log(np.sinh(half_d)))
#     log_term2 = np.sum(np.log(d) - np.log(2.0) - log_sinh)
#     return float(np.exp(np.clip(log_term1 + log_term2, -700, 700)) )

# ─────────────────────────────────────────────
# Transformer
# ─────────────────────────────────────────────

class CauchyKernelTransformer(BaseEstimator, TransformerMixin):
    """
    Scikit-learn transformer that maps a set of SPD matrices to a Gram-matrix
    column using the Cauchy kernel on the SPD manifold.

    Parameters
    ----------
    kappa : float, default=1.0
        Bandwidth parameter of the Cauchy kernel.  Larger values make the
        kernel broader (less discriminative); smaller values make it sharper.

    Attributes
    ----------
    X_train_ : ndarray of shape (m, p, p)
        Training SPD matrices stored during ``fit``.

    Notes
    -----
    Input arrays must contain symmetric positive definite matrices.
    A small eigenvalue clip (1e-12) is applied internally for robustness.

    Usage
    -----
    Drop-in replacement for ``SteinKernelTransformer`` inside a
    ``Pipeline`` + ``GridSearchCV`` workflow::

        pipe = Pipeline([
            ("kernel", CauchyKernelTransformer()),
            ("svm",    SVC(kernel="precomputed")),
        ])
        grid = GridSearchCV(pipe, {"kernel__kappa": [0.5, 1.0, 2.0]})
        grid.fit(X_train, y_train)
    """

    def __init__(self, kappa: float = 1.0):
        self.kappa = kappa
        self.X_train_ = None

    # ------------------------------------------------------------------
    def fit(self, X: np.ndarray, y=None):
        """Store training SPD matrices.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_channels, n_channels)
        y : ignored
        """
        self.X_train_ = X
        return self

    # ------------------------------------------------------------------
    def transform(self, X: np.ndarray) -> np.ndarray:
        """Compute the Cauchy kernel matrix K[i, j] = k(X[i], X_train[j]).

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_channels, n_channels)

        Returns
        -------
        K : ndarray of shape (n_samples, n_train_samples)
        """
        if self.X_train_ is None:
            raise ValueError("Call fit() before transform().")

        N = X.shape[0]
        M = self.X_train_.shape[0]
        K = np.zeros((N, M))

        for i in range(N):
            for j in range(M):
                K[i, j] = _cauchy_kernel_single(X[i], self.X_train_[j], kappa=self.kappa)

        return K


# ─────────────────────────────────────────────
# Quick smoke-test
# ─────────────────────────────────────────────

# if __name__ == "__main__":
#     rng = np.random.default_rng(42)

#     def _random_spd(n, k):
#         """Return k random (n x n) SPD matrices."""
#         out = []
#         for _ in range(k):
#             A = rng.standard_normal((n, n))
#             out.append(A @ A.T + np.eye(n))
#         return np.array(out)

#     n_ch = 4
#     X_tr = _random_spd(n_ch, 20)
#     X_te = _random_spd(n_ch, 8)
#     y_tr = rng.integers(0, 2, size=20)

#     pipe = Pipeline([
#         ("kernel", CauchyKernelTransformer()),
#         ("svm",    SVC(kernel="precomputed")),
#     ])

#     param_grid = {"kernel__kappa": [0.5, 1.0, 2.0]}
#     gs = GridSearchCV(pipe, param_grid, cv=4, scoring="accuracy", n_jobs=-1)
#     gs.fit(X_tr, y_tr)

#     print("Best kappa :", gs.best_params_["kernel__kappa"])
#     print(f"Best CV acc: {gs.best_score_:.4f}")
#     print("Test  preds:", gs.predict(X_te))

In [ ]:
"""
cauchy_kernel_transformer.py
============================

Scikit-learn transformer implementing the strictly positive-definite Cauchy
kernel on the manifold of Symmetric Positive Definite (SPD) matrices, together
with the Chebyshev utility functions verified in
``chebyshev_cauchy_verification_notebook.ipynb``.

Kernel back-ends
----------------
Two proper kernel back-ends are provided:

    method='full'   (default, recommended)
        K(X,Y) = exp(
            sum_i  log[ kappa^2 / (kappa^2 + s_i^2) ]
          + sum_{i<j}  log[ |s_j - s_i| / (2 sinh(|s_j - s_i| / 2)) ]
        )
        The pairwise sinh-correction encodes the Vandermonde factor from the
        Helgason-Fourier / spherical-transform derivation.  Evaluated via
        ``log_sinh_ratio`` which switches to a Taylor series for near-zero
        gaps, making it stable in both the well-separated and near-collision
        regimes.  Symmetric, strictly positive-definite for all n >= 1.

    method='simple'
        K(X,Y) = prod_i  kappa^2 / (kappa^2 + s_i^2)
        The diagonal spectral contribution only; no Vandermonde correction.
        Fast and symmetric but does not encode the full Riemannian geometry.

In both cases s_i = log lam_i(X^{-1/2} Y X^{-1/2}) are the generalised
log-eigenvalues, computed via ``scipy.linalg.eigh``.

Chebyshev derivative utilities
--------------------------------
The module also exports the building blocks verified in the notebook:

    ``chebyshev_U(n, x)``
        U_n(x) via the three-term recurrence.

    ``chebyshev_U_matrix(n_rows, x)``
        Vandermonde-style matrix T[k, j] = U_k(x_j).

    ``cauchy_derivative_row(k, s, kappa)``
        Closed-form k-th derivative of gamma(s) = 1/(s^2+kappa^2) via U_{k-1}.

    ``cauchy_derivative_matrix(s, kappa)``
        Full N x N matrix A[k, l] = d^k/ds^k gamma(s_l) built from the
        Chebyshev recurrence.  This is a sympy-free replacement for the
        derivative cache in the original implementation, retained here as an
        analysis / verification utility matching the notebook.

    ``log_sinh_ratio(delta)``
        Numerically stable log(d / (2 sinh(d/2))).

    ``generalized_log_spectrum(X, Y)``
        log-eigenvalues of Y w.r.t. X via ``scipy.linalg.eigh``.

Mathematical basis
------------------
For X, Y in P_n, the generalised log-eigenvalues s_i = log lam_i(Y, X) are
the log-ratios of the SPD spectra.  Under the spherical-transform framework
(Godement's theorem), the spectral density

    gamma(t) = kappa^2 / (kappa^2 + t^2)

yields a strictly positive-definite zonal function for all n >= 1.  The
sinh-correction term comes from the Vandermonde-type factor in the
Helgason-Fourier inversion formula and is evaluated stably via log_sinh_ratio.

The Chebyshev closed form for derivatives of gamma is

    d^{k-1}/ds^{k-1} [1/(s^2+kappa^2)]
    = (-1)^{k+1} (k-1)! (s^2+kappa^2)^{-(k+1)/2}
      U_{k-1}( s / sqrt(s^2+kappa^2) )

where U_n satisfies U_{n+1}(x) = 2x U_n(x) - U_{n-1}(x).  This replaces
the sympy symbolic differentiation in the original implementation with an
O(N^2) recurrence that is allocation-free at evaluation time.

Note on the determinant-based formula
--------------------------------------
The original ``_cauchy_kernel_single`` attempted to evaluate the kernel as

    K = det(Z)^{(N-1)/2}  *  |det M(s)|  /  |V(s)|

where Z = X^{-1/2} Y X^{-1/2} and M is the derivative matrix.  This formula
is NOT symmetric in X, Y because swapping X <-> Y negates all s_i, which
flips the sign of the det(Z)^{(N-1)/2} factor while leaving |det M| and
|V(s)| unchanged.  Dropping the det-power restores symmetry but the
resulting Gram matrix is not PSD.  The correct kernel is the sinh-based
``method='full'`` formula; ``cauchy_derivative_matrix`` is retained as an
analysis tool.

References
----------
Verified symbolically (SymPy) and numerically (mpmath, dps=80) in
``chebyshev_cauchy_verification_notebook.ipynb``.
"""

from __future__ import annotations

import math
from typing import Literal

import numpy as np
from scipy.linalg import eigh
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted


__all__ = [
    # Kernel transformer
    "CauchyKernelTransformer",
    # Chebyshev utilities (public, verified in notebook)
    "chebyshev_U",
    "chebyshev_U_matrix",
    "cauchy_derivative_row",
    "cauchy_derivative_matrix",
    "log_sinh_ratio",
    "generalized_log_spectrum",
]


# =============================================================================
# Chebyshev utilities (sympy-free, verified in notebook)
# =============================================================================

def chebyshev_U(n: int, x: np.ndarray) -> np.ndarray:
    """
    Evaluate U_n(x) via the three-term recurrence

        U_{n+1}(x) = 2x U_n(x) - U_{n-1}(x),   U_0 = 1,  U_1 = 2x.

    Parameters
    ----------
    n : int >= 0
    x : array_like, arbitrary shape

    Returns
    -------
    ndarray of same shape as x
    """
    x = np.asarray(x, dtype=np.float64)
    if n == 0:
        return np.ones_like(x)
    if n == 1:
        return 2.0 * x
    U_prev, U_curr = np.ones_like(x), 2.0 * x
    for _ in range(1, n):
        U_prev, U_curr = U_curr, 2.0 * x * U_curr - U_prev
    return U_curr


def chebyshev_U_matrix(n_rows: int, x: np.ndarray) -> np.ndarray:
    """
    Build the (n_rows x m) Chebyshev-U Vandermonde matrix

        T[k, j] = U_k(x[j]),   k = 0, ..., n_rows - 1.

    Uses the in-place recurrence; at most two rows are allocated simultaneously.

    Parameters
    ----------
    n_rows : int >= 1
    x      : 1-D array of evaluation points, length m

    Returns
    -------
    T : ndarray of shape (n_rows, m)
    """
    x = np.asarray(x, dtype=np.float64)
    m = x.size
    T = np.empty((n_rows, m), dtype=np.float64)
    T[0, :] = 1.0
    if n_rows > 1:
        T[1, :] = 2.0 * x
    for k in range(2, n_rows):
        T[k, :] = 2.0 * x * T[k - 1, :] - T[k - 2, :]
    return T


def cauchy_derivative_row(k: int, s: np.ndarray, kappa: float) -> np.ndarray:
    """
    Evaluate the (k-1)-th derivative of gamma(s) = 1 / (s^2 + kappa^2) at
    every point in ``s`` using the Chebyshev closed form

        gamma^{(k-1)}(s) = (-1)^{k+1} (k-1)! (s^2+kappa^2)^{-(k+1)/2}
                           U_{k-1}( s / sqrt(s^2+kappa^2) ).

    No symbolic computation is performed; U_{k-1} is evaluated via the
    three-term Chebyshev recurrence.

    Parameters
    ----------
    k     : int >= 1  (derivative order is k-1; k=1 gives gamma itself)
    s     : 1-D array of evaluation points
    kappa : float > 0

    Returns
    -------
    ndarray of same shape as s
    """
    s = np.asarray(s, dtype=np.float64)
    r2 = s * s + kappa * kappa
    x  = s / np.sqrt(r2)
    return ((-1.0) ** (k + 1)
            * math.factorial(k - 1)
            * r2 ** (-(k + 1) / 2.0)
            * chebyshev_U(k - 1, x))


def cauchy_derivative_matrix(s: np.ndarray, kappa: float) -> np.ndarray:
    """
    Build the N x N derivative matrix

        A[k, l] = d^k/ds^k  [1/(s_l^2+kappa^2)],   k, l = 0, ..., N-1

    using the Chebyshev closed form row-by-row.

    This is a sympy-free drop-in for the derivative cache in the original
    ``_cauchy_kernel_single``.  It is used in the notebook for the
    Chebyshev-Vandermonde determinant analysis (Cells 29-35) and is retained
    here as a verification / analysis utility.

    .. note::
        This matrix is **not** used internally by the kernel transformer.
        The 'full' method uses the sinh-based formula which is numerically
        more robust.  See module docstring for the symmetry issue with the
        determinant-based kernel formula.

    Parameters
    ----------
    s     : 1-D array of log-eigenvalues, length N
    kappa : float > 0

    Returns
    -------
    A : ndarray of shape (N, N)
    """
    s = np.asarray(s, dtype=np.float64)
    n = s.size
    r2 = s * s + kappa * kappa
    x  = s / np.sqrt(r2)
    T  = chebyshev_U_matrix(n, x)   # T[k, l] = U_k(x_l)

    A = np.empty((n, n), dtype=np.float64)
    for k in range(n):
        # 0-indexed: row k is d^k/ds^k gamma(s_l)
        # closed form: (-1)^{k+2} * k! * r2^{-(k+2)/2} * U_k(x)
        sign = (-1.0) ** (k + 2)
        A[k, :] = sign * math.factorial(k) * r2 ** (-(k + 2) / 2.0) * T[k, :]
    return A


# =============================================================================
# SPD spectral utilities
# =============================================================================

def generalized_log_spectrum(
    X: np.ndarray,
    Y: np.ndarray,
    jitter: float = 1e-10,
) -> np.ndarray:
    """
    Compute the generalised log-eigenvalues

        s_i = log lam_i(X^{-1/2} Y X^{-1/2})

    via ``scipy.linalg.eigh(Y + jitter I, X + jitter I)``.

    Parameters
    ----------
    X, Y   : ndarray of shape (n, n), SPD
    jitter : small regularisation added to both matrices

    Returns
    -------
    s : ndarray of shape (n,), unsorted log-eigenvalues
    """
    n = X.shape[0]
    I = np.eye(n)
    lam = eigh(Y + jitter * I, X + jitter * I, eigvals_only=True)
    return np.log(np.maximum(lam, 1e-300))


def log_sinh_ratio(delta: np.ndarray, tol: float = 1e-8) -> np.ndarray:
    """
    Numerically stable evaluation of log( d / (2 sinh(d/2)) ) for d >= 0.

    For d < tol the Taylor expansion

        log(d / (2 sinh(d/2))) = -d^2/24 + d^4/2880 + O(d^6)

    avoids catastrophic cancellation; for d >= tol the exact formula is used.

    Both cases are computed in a single vectorised pass over ``delta``.

    Parameters
    ----------
    delta : array_like of non-negative floats (pairwise spectral gaps)
    tol   : float, crossover threshold (default 1e-8)

    Returns
    -------
    ndarray of same shape as delta
    """
    delta = np.asarray(delta, dtype=np.float64)
    out = np.empty_like(delta)
    small = delta < tol
    d = delta[small]
    out[small] = -(d * d) / 24.0 + (d ** 4) / 2880.0
    d = delta[~small]
    out[~small] = np.log(d) - np.log(2.0 * np.sinh(d / 2.0))
    return out


# =============================================================================
# Internal per-pair kernel evaluators
# =============================================================================

def _kernel_simple(X: np.ndarray, Y: np.ndarray, kappa: float, jitter: float) -> float:
    """
    Simple Cauchy kernel: product of scalar Cauchy evaluations over the
    generalised log-spectrum.

        K(X,Y) = prod_i  kappa^2 / (kappa^2 + s_i^2)

    Symmetric, bounded in (0, 1].  No pairwise Vandermonde correction.
    """
    s = generalized_log_spectrum(X, Y, jitter=jitter)
    return float(np.prod((kappa * kappa) / (kappa * kappa + s * s)))


def _kernel_full(X: np.ndarray, Y: np.ndarray, kappa: float, jitter: float) -> float:
    """
    Full Cauchy kernel with sinh-based pairwise Vandermonde correction.

        K(X,Y) = exp(
            sum_i  log[ kappa^2 / (kappa^2 + s_i^2) ]
          + sum_{i<j}  log[ |s_j - s_i| / (2 sinh(|s_j - s_i| / 2)) ]
        )

    Both log-sums are <= 0, so K is always in (0, 1].  Symmetric and PSD.
    Numerically stable in the near-collision regime via ``log_sinh_ratio``.
    """
    s = np.sort(generalized_log_spectrum(X, Y, jitter=jitter))
    logk = float(np.sum(np.log((kappa * kappa) / (kappa * kappa + s * s))))
    n = s.size
    if n > 1:
        idx_i, idx_j = np.tril_indices(n, k=-1)
        gaps = np.abs(s[idx_j] - s[idx_i])   # all N(N-1)/2 pairwise gaps
        logk += float(np.sum(log_sinh_ratio(gaps)))
    return float(np.exp(np.clip(logk, -700.0, 700.0)))


_KERNEL_FN: dict = {
    "simple": _kernel_simple,
    "full":   _kernel_full,
}


# =============================================================================
# Scikit-learn transformer
# =============================================================================

class CauchyKernelTransformer(BaseEstimator, TransformerMixin):
    """
    Scikit-learn transformer that builds the (n_test x n_train) Gram matrix
    for the strictly positive-definite Cauchy kernel on the SPD manifold.

    For X, Y in P_n the kernel is computed from the generalised log-eigenvalues
    s_i = log lam_i(X^{-1/2} Y X^{-1/2}), using ``scipy.linalg.eigh``.

    Parameters
    ----------
    kappa : float, default=1.0
        Bandwidth parameter; must be > 0.  The kernel spectral density decays
        as (pi/kappa) exp(-kappa|omega|), so kappa is an *inverse* bandwidth:

          - smaller kappa  ->  broader spectrum, higher expressivity
          - larger  kappa  ->  narrower spectrum, stronger smoothing

    method : {'full', 'simple'}, default='full'
        Kernel computation back-end.

        ``'full'``
            Product kernel plus sinh-based pairwise Vandermonde correction.
            Numerically stable across separated, clustered, and small-scale
            spectral regimes.  Strictly PSD for all matrix dimensions n >= 1.
            **Recommended for all practical use.**

        ``'simple'``
            Product kernel only; no Vandermonde correction.
            K(X,Y) = prod_i kappa^2/(kappa^2+s_i^2).
            Fastest; suitable for n=1 or as a diagnostic baseline.

    jitter : float, default=1e-10
        Small eps added to both matrices before the generalised eigenproblem,
        guarding against near-singularity.

    Attributes
    ----------
    X_train_ : ndarray of shape (n_train, p, p)
        Training SPD matrices stored during ``fit``.

    n_features_in_ : int
        Matrix dimension p observed at fit time.

    Examples
    --------
    >>> from sklearn.pipeline import make_pipeline
    >>> from sklearn.svm import SVC
    >>> pipe = make_pipeline(
    ...     CauchyKernelTransformer(kappa=1.0, method='full'),
    ...     SVC(kernel='precomputed'),
    ... )
    >>> pipe.fit(X_train_cov, y_train)   # X_train_cov: (n_train, p, p) SPD
    >>> pipe.predict(X_test_cov)         # X_test_cov:  (n_test,  p, p) SPD
    """

    def __init__(
        self,
        kappa: float = 1.0,
        method: Literal["full", "simple"] = "full",
        jitter: float = 1e-10,
    ) -> None:
        if kappa <= 0:
            raise ValueError(f"kappa must be > 0, got {kappa!r}")
        if method not in _KERNEL_FN:
            raise ValueError(
                f"method must be one of {list(_KERNEL_FN)!r}, got {method!r}"
            )
        self.kappa  = kappa
        self.method = method
        self.jitter = jitter

    # ------------------------------------------------------------------
    def fit(self, X: np.ndarray, y=None) -> "CauchyKernelTransformer":
        """
        Store training covariance matrices.

        Parameters
        ----------
        X : ndarray of shape (n_train, p, p)
            Batch of SPD covariance matrices.
        y : ignored

        Returns
        -------
        self
        """
        X = np.asarray(X, dtype=np.float64)
        if X.ndim != 3 or X.shape[1] != X.shape[2]:
            raise ValueError(
                "X must have shape (n_samples, p, p); "
                f"got {X.shape}"
            )
        self.X_train_        = X.copy()
        self.n_features_in_  = X.shape[1]
        return self

    # ------------------------------------------------------------------
    def transform(self, X: np.ndarray) -> np.ndarray:
        """
        Compute the (n_test x n_train) kernel Gram matrix.

        Parameters
        ----------
        X : ndarray of shape (n_test, p, p)
            Test SPD covariance matrices.

        Returns
        -------
        K : ndarray of shape (n_test, n_train)
            K[i, j] = kernel(X[i], X_train_[j]).
        """
        check_is_fitted(self, "X_train_")
        X = np.asarray(X, dtype=np.float64)
        if X.ndim != 3 or X.shape[1] != X.shape[2]:
            raise ValueError(
                "X must have shape (n_samples, p, p); "
                f"got {X.shape}"
            )
        if X.shape[1] != self.n_features_in_:
            raise ValueError(
                f"Expected p={self.n_features_in_} matrices; "
                f"got {X.shape[1]}x{X.shape[2]}"
            )

        kernel_fn = _KERNEL_FN[self.method]
        n_test  = X.shape[0]
        n_train = self.X_train_.shape[0]
        K = np.empty((n_test, n_train), dtype=np.float64)

        for i in range(n_test):
            for j in range(n_train):
                K[i, j] = kernel_fn(
                    X[i], self.X_train_[j], self.kappa, self.jitter
                )
        return K

    # ------------------------------------------------------------------
    def fit_transform(self, X: np.ndarray, y=None) -> np.ndarray:
        """
        Fit on X and return the (n_train x n_train) training Gram matrix.

        Equivalent to ``self.fit(X).transform(X)`` but avoids a copy.

        Parameters
        ----------
        X : ndarray of shape (n_train, p, p)
        y : ignored

        Returns
        -------
        K : ndarray of shape (n_train, n_train)
        """
        return self.fit(X, y).transform(X)

In [2]:
import numpy as np
from tqdm.auto import tqdm
from sklearn.svm import SVC
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, ShuffleSplit
from pyriemann.estimation import Covariances
from moabb.datasets import BNCI2014001
from moabb.paradigms import MotorImagery
import mne
import sympy as sp

# ── Butterworth paradigm ───────────────────────────────────────────────────────

class ButterworthMotorImagery(MotorImagery):
    def preprocess_raw(self, raw, dataset, fitting_config=None):
        iir_params = dict(order=5, ftype='butter')
        raw.filter(l_freq=self.fmin, h_freq=self.fmax,
                   method='iir', iir_params=iir_params, verbose=False)
        return super().preprocess_raw(raw, dataset, fitting_config)

# ── MicrovoltScaler ────────────────────────────────────────────────────────────

class MicrovoltScaler(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X): return X * 1e6

# ── Stein kernel ───────────────────────────────────────────────────────────────

class SteinGramMatrix(BaseEstimator, TransformerMixin):
    def __init__(self, normalized=False, beta=0.5):
        self.normalized = normalized
        self.beta = beta
        self.X_train_ = None

    def fit(self, X, y=None):
        self.X_train_ = X
        return self

    def transform(self, X):
        N, M = len(X), len(self.X_train_)
        K = np.zeros((N, M))
        if self.normalized:
            log_det_X     = np.array([np.linalg.slogdet(x)[1] for x in X])
            log_det_train = np.array([np.linalg.slogdet(t)[1] for t in self.X_train_])
        for i in range(N):
            for j in range(M):
                M_ij = (X[i] + self.X_train_[j]) / 2.0
                _, log_det_M = np.linalg.slogdet(M_ij)
                if self.normalized:
                    log_K_ij = (-self.beta * log_det_M
                                + (self.beta / 2.0) * log_det_X[i]
                                + (self.beta / 2.0) * log_det_train[j])
                else:
                    log_K_ij = -self.beta * log_det_M
                K[i, j] = np.exp(log_K_ij)
        return K


<frozen importlib._bootstrap>:219: FutureWarning: mne.io.pick.pick_types is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:219: FutureWarning: mne.io.pick.pick_channels_regexp is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:219: FutureWarning: mne.io.pick.channel_type is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
/Users/rishabhkumar/miniconda3/envs/kernel-experiments/lib/python3.8/site-packages/moabb/pipelines/__init__.py:26: ModuleNotFoundError: Tensorflow is not installed. You won't be able to use these MOABB pipelines if you attempt to do so.
  warn(


In [3]:
import threading
import joblib
from tqdm.notebook import tqdm
from sklearn.model_selection import GridSearchCV, ParameterGrid


class TqdmGridSearchCV(GridSearchCV):
    def fit(self, X, y=None, **fit_params):
        n_candidates = len(ParameterGrid(self.param_grid))
        n_splits = self.cv.get_n_splits(X, y)
        total_fits = n_candidates * n_splits

        old_cb = joblib.parallel.BatchCompletionCallBack
        pbar = tqdm(total=total_fits, desc="GridSearchCV", leave=False)

        class _TqdmBatchCallback(old_cb):
            _lock = threading.Lock()

            def __call__(self, *args, **kwargs):
                with self._lock:
                    pbar.update(self.batch_size)
                return super().__call__(*args, **kwargs)

        joblib.parallel.BatchCompletionCallBack = _TqdmBatchCallback
        try:
            return super().fit(X, y, **fit_params)
        finally:
            joblib.parallel.BatchCompletionCallBack = old_cb
            pbar.close()

In [11]:
from sklearn.model_selection import ParameterGrid
import joblib

dataset  = BNCI2014001()
paradigm = ButterworthMotorImagery(fmin=8.0, fmax=30.0, tmin=0.5, tmax=2.5)

cv_strategy = ShuffleSplit(n_splits=30, test_size=0.2, random_state=42)

# Separate param grids — C for SVC, kappa for Cauchy
# param_grid_svc    = {'svc__C': [0.1, 1, 10, 100, 1000]}
# param_grid_cauchy = {'svc__C': [0.1, 1, 10, 100, 1000],
#                      'cauchykerneltransformer__kappa': [0.1, 0.5, 1.0, 2.0, 5.0]}

param_grid_cauchy = {'svc__C': [0.1, 1, 10, 100, 1000],
                     'cauchykerneltransformer__kappa': [0.1, 0.5, 1.0, 2.0, 5.0]}

# pipeline_std    = make_pipeline(Covariances(estimator='scm'),
#                                 SteinGramMatrix(normalized=False),
#                                 SVC(kernel='precomputed'))

# pipeline_norm   = make_pipeline(Covariances(estimator='scm'),
#                                 SteinGramMatrix(normalized=True),
#                                 SVC(kernel='precomputed'))

pipeline_cauchy = make_pipeline(Covariances(estimator='scm'),
                                CauchyKernelTransformer(kappa=1.0),
                                SVC(kernel='precomputed'))

# grid_std    = GridSearchCV(pipeline_std,    param_grid_svc,    cv=cv_strategy, n_jobs=-1)
# grid_norm   = GridSearchCV(pipeline_norm,   param_grid_svc,    cv=cv_strategy, n_jobs=-1)


grid_cauchy = TqdmGridSearchCV(
    pipeline_cauchy,
    param_grid_cauchy,
    cv=cv_strategy,
    n_jobs=-1
)

# ── Evaluation loop ────────────────────────────────────────────────────────────

results_std, results_norm, results_cauchy = [], [], []
subjects = [1, 2, 3, 4, 5, 6, 7, 8, 9]

for subject in tqdm(subjects, desc='Processing subjects'):
    X, y, metadata = paradigm.get_data(dataset, subjects=[subject])

    train_idx = metadata['session'] == '0train'
    test_idx  = metadata['session'] == '1test'
    X_train, y_train = X[train_idx], y[train_idx]
    X_test,  y_test  = X[test_idx],  y[test_idx]

    grid_cauchy.fit(X_train, y_train)

    # ── Explicitly apply best params back to the pipeline ─────────────────
    best_params = grid_cauchy.best_params_
    pipeline_cauchy.set_params(**best_params)          # kappa + C updated
    pipeline_cauchy.fit(X_train, y_train)              # refit with best params
    score = pipeline_cauchy.score(X_test, y_test)      # score using best model
    # ──────────────────────────────────────────────────────────────────────

    results_cauchy.append(score)

    print(f"Sub {subject:02d}  "
          f"Cauchy: {results_cauchy[-1]*100:5.2f}%  "
          f"[best kappa={best_params['cauchykerneltransformer__kappa']}  "
          f"C={best_params['svc__C']}]")

# print(f"\n{'─'*60}")
# print(f"Mean  Stein:      {np.mean(results_std)*100:.2f}%  ± {np.std(results_std)*100:.2f}%")
# print(f"Mean  Stein-Norm: {np.mean(results_norm)*100:.2f}%  ± {np.std(results_norm)*100:.2f}%")
print(f"Mean  Cauchy:     {np.mean(results_cauchy)*100:.2f}%  ± {np.std(results_cauchy)*100:.2f}%")

BNCI2014001 has been renamed to BNCI2014_001. BNCI2014001 will be removed in version 1.1.
The dataset class name 'BNCI2014001' must be an abbreviation of its code 'BNCI2014-001'. See moabb.datasets.base.is_abbrev for more information.
Choosing from all possible events


Processing subjects:   0%|          | 0/9 [00:00<?, ?it/s]

/Users/rishabhkumar/miniconda3/envs/kernel-experiments/lib/python3.8/site-packages/sklearn/preprocessing/_function_transformer.py:312: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/Users/rishabhkumar/miniconda3/envs/kernel-experiments/lib/python3.8/site-packages/sklearn/preprocessing/_function_transformer.py:312: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/Users/rishabhkumar/miniconda3/envs/kernel-experiments/lib/python3.8/site-packages/sklearn/preprocessing/_function_transformer.py:312: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/Users/rishabhkumar/miniconda3/envs/k

GridSearchCV:   0%|          | 0/750 [00:00<?, ?it/s]

Sub 01  Cauchy: 81.94%  [best kappa=0.5  C=10]


...

GridSearchCV:   0%|          | 0/750 [00:00<?, ?it/s]

Sub 08  Cauchy: 79.51%  [best kappa=5.0  C=10]


/Users/rishabhkumar/miniconda3/envs/kernel-experiments/lib/python3.8/site-packages/sklearn/preprocessing/_function_transformer.py:312: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/Users/rishabhkumar/miniconda3/envs/kernel-experiments/lib/python3.8/site-packages/sklearn/preprocessing/_function_transformer.py:312: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/Users/rishabhkumar/miniconda3/envs/kernel-experiments/lib/python3.8/site-packages/sklearn/preprocessing/_function_transformer.py:312: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/Users/rishabhkumar/miniconda3/envs/k

GridSearchCV:   0%|          | 0/750 [00:00<?, ?it/s]

Sub 09  Cauchy: 80.56%  [best kappa=2.0  C=10]
Mean  Cauchy:     65.82%  ± 15.28%


In [14]:
import numpy as np

scores = np.array(results_cauchy, dtype=float) * 100.0
score_cols = " & ".join(f"{x:.2f}" for x in scores)
mean_score = np.mean(scores)
std_score = np.std(scores)

latex_row = f"& Cauchy Kernel & {score_cols} & {mean_score:.2f} & {std_score:.2f} \\\\"
print(latex_row)
print(results_cauchy)

& Cauchy Kernel & 81.94 & 46.18 & 79.51 & 53.12 & 48.96 & 47.57 & 75.00 & 79.51 & 80.56 & 65.82 & 15.28 \\
[0.8194444444444444, 0.4618055555555556, 0.7951388888888888, 0.53125, 0.4895833333333333, 0.4756944444444444, 0.75, 0.7951388888888888, 0.8055555555555556]


In [ ]:
from scipy.linalg import eigh as scipy_eigh

# Small BCI sample -> SPD covariances
kappa = 1.0
n_small = 
X_cov = Covariances(estimator="scm").fit_transform(X_train[:n_small])

def _log_k_cauchy(X, Y, kappa=1.0, drop_sinh=False):
    rho = scipy_eigh(Y, X, eigvals_only=True)
    s = np.sort(np.log(np.clip(rho, 1e-12, None)))

    log_term1 = np.sum(2.0 * np.log(kappa) - np.log(kappa**2 + s**2))

    i_idx, j_idx = np.triu_indices(len(s), k=1)
    d = s[j_idx] - s[i_idx]
    d = d[d > 1e-10]

    if d.size == 0:
        return log_term1

    if drop_sinh:
        # no curvature correction
        log_term2 = np.sum(np.log(d) - np.log(2.0))
    else:
        half_d = d / 2.0
        log_sinh = np.where(
            half_d > 20.0,
            half_d - np.log(2.0),
            np.log(np.sinh(half_d))
        )
        log_term2 = np.sum(np.log(d) - np.log(2.0) - log_sinh)

    return log_term1 + log_term2

def _gram(Xs, kappa=1.0, drop_sinh=False):
    n = len(Xs)
    K = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            v = np.exp(np.clip(_log_k_cauchy(Xs[i], Xs[j], kappa, drop_sinh), -700, 700))
            K[i, j] = K[j, i] = v
    return K

def _normalize_gram(K):
    d = np.sqrt(np.clip(np.diag(K), 1e-12, None))
    return K / np.outer(d, d)



K_full = _gram(X_cov, kappa=kappa, drop_sinh=False)
K_drop = _gram(X_cov, kappa=kappa, drop_sinh=True)

Kf = _normalize_gram(K_full)
Kd = _normalize_gram(K_drop)

iu = np.triu_indices_from(Kf, k=1)
corr = np.corrcoef(Kf[iu], Kd[iu])[0, 1]
rel_fro = np.linalg.norm(Kf - Kd, ord="fro") / np.linalg.norm(Kf, ord="fro")

eig_full = np.linalg.eigvalsh((Kf + Kf.T) / 2.0)
eig_drop = np.linalg.eigvalsh((Kd + Kd.T) / 2.0)

print(f"n_small={n_small}, kappa={kappa}")
print(f"off-diagonal corr(full vs no-sinh): {corr:.6f}")
print(f"relative Frobenius diff          : {rel_fro:.6f}")
print(f"min eigenvalue (full)            : {eig_full.min():.3e}")
print(f"min eigenvalue (no-sinh)         : {eig_drop.min():.3e}")

n_small=404, kappa=1.0
off-diagonal corr(full vs no-sinh): -0.000142
relative Frobenius diff          : 0.000000
min eigenvalue (full)            : 1.000e+00
min eigenvalue (no-sinh)         : 1.000e+00


In [6]:
pipeline_cauchy = make_pipeline(Covariances(estimator='scm'),
                                CauchyKernelTransformer(kappa=999),  # obviously wrong value
                                SVC(kernel='precomputed'))

# Print the parameters of the cauchy pipeline
for param, value in sorted(pipeline_cauchy.get_params().items()):
    print(f"{param}: {value}")

    # Check that kappa and C can be changed after pipeline initialization
    rng = np.random.default_rng(42)

    for i in range(5):
        kappa_rand = float(rng.uniform(0.05, 10.0))
        c_rand = float(10 ** rng.uniform(-2, 3))  # ~[0.01, 1000]

        pipeline_cauchy.set_params(
            cauchykerneltransformer__kappa=kappa_rand,
            svc__C=c_rand
        )

        # Verify params were updated
        p = pipeline_cauchy.get_params()
        print(
            f"Trial {i+1}: "
            f"set kappa={kappa_rand:.4f}, C={c_rand:.4f} | "
            f"stored kappa={p['cauchykerneltransformer__kappa']:.4f}, "
            f"stored C={p['svc__C']:.4f}"
        )

        # Refit and score with the updated params
        pipeline_cauchy.fit(X_train, y_train)
        score = pipeline_cauchy.score(X_test, y_test)
        print(f"         test score={score:.4f}")

cauchykerneltransformer: CauchyKernelTransformer(kappa=999)
Trial 1: set kappa=7.7509, C=1.5646 | stored kappa=7.7509, stored C=1.5646
         test score=0.6771
Trial 2: set kappa=8.5930, C=30.6789 | stored kappa=8.5930, stored C=30.6789
         test score=0.7882
Trial 3: set kappa=0.9871, C=755.2866 | stored kappa=0.9871, stored C=755.2866
         test score=0.7743
Trial 4: set kappa=7.6233, C=85.1768 | stored kappa=7.6233, stored C=85.1768
         test score=0.7986
Trial 5: set kappa=1.3247, C=1.7862 | stored kappa=1.3247, stored C=1.7862
         test score=0.7847
cauchykerneltransformer__kappa: 999
Trial 1: set kappa=7.7509, C=1.5646 | stored kappa=7.7509, stored C=1.5646
         test score=0.6771
Trial 2: set kappa=8.5930, C=30.6789 | stored kappa=8.5930, stored C=30.6789
         test score=0.7882
Trial 3: set kappa=0.9871, C=755.2866 | stored kappa=0.9871, stored C=755.2866
         test score=0.7743
Trial 4: set kappa=7.6233, C=85.1768 | stored kappa=7.6233, stored C=85.17

KeyboardInterrupt: 